In [11]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------------
# Parameters and File Paths
# -------------------------------
smiles_col = 'OriginalSmiles'
target = 'pKa'
data_file = "./dw_data/Opt1_acidic_tr.csv"  # using the training file as data source

# Read the dataset once.
data_df = pd.read_csv(data_file)

# Initialize the salt remover.
saltRemover = SaltRemover(defnFilename='./Salts.txt')

# Define the grid for fingerprint parameters.
fp_radii = [1, 2, 3, 4]
fp_sizes = [10, 128, 256, 512, 1024]

# Define the hyperparameter grid for XGBoost.
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

# Prepare a list to store performance results.
results = []

# -------------------------------
# Loop over fingerprint parameters.
# -------------------------------
for radius in fp_radii:
    for size in fp_sizes:
        print(f"\nEvaluating: Radius = {radius}, Fingerprint Size = {size}")
        try:
            # Generate RDKit molecules from SMILES.
            rdkit_mols = data_df[smiles_col].astype(str).apply(Chem.MolFromSmiles)
            # Remove salts.
            rdkit_mols = rdkit_mols.apply(saltRemover.StripMol)
            # Generate fingerprints as bit strings with current radius and size.
            fps = rdkit_mols.apply(lambda mol: AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=size).ToBitString())
            # Convert bit strings into a numerical array.
            X = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fps]).astype(float)
        except Exception as e:
            print(f"Error generating fingerprints for radius {radius} and size {size}: {e}")
            continue
        
        # Target values.
        y = data_df[target].values
        
        # Split into training and test sets.
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11)
        
        print(f'Shape (X, y): ({X.shape}, {y.shape})')
        print(f'Shape (X_train, y_train): ({X_train.shape}, {y_train.shape})')
        print(f'Shape (X_test, y_test): ({X_test.shape}, {y_test.shape})')
        
        # Define the base XGBoost model.
        xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
        
        # Set up GridSearchCV for hyperparameter tuning.
        grid_search = GridSearchCV(
            estimator=xgbr,
            param_grid=param_grid,
            scoring='r2',
            cv=5,
            n_jobs=5,
            verbose=2
        )
        
        # Fit the grid search on the training data.
        grid_search.fit(X_train, y_train)
        
        # Extract the best model.
        best_xgbr_model = grid_search.best_estimator_
        
        # Evaluate on training and test data.
        y_pred_train = best_xgbr_model.predict(X_train)
        y_pred_test = best_xgbr_model.predict(X_test)
        
        train_r2 = r2_score(y_train, y_pred_train)
        test_r2 = r2_score(y_test, y_pred_test)
        
        # Collect the performance metrics.
        results.append({
            "Radius": radius,
            "fingerprint_size": size,
            "Train R2": train_r2,
            "Test R2": test_r2,
            "Best Params": grid_search.best_params_,
            "model": "XGB"
        })
        
        print(f"Best Params: {grid_search.best_params_}")
        print(f"Train R2: {train_r2:.4f}, Test R2: {test_r2:.4f}")

# Convert results to a DataFrame.
results_df = pd.DataFrame(results)

# Save the results to a CSV file.
results_df.to_csv("performance_all_radii.csv", index=False)
print("\nPerformance metrics saved to 'performance_all_radii.csv'.")

# -------------------------------
# Plotting the results.
# -------------------------------
# Create a faceted line plot for Test R2 vs fingerprint size, faceted by Radius.
g = sns.relplot(
    data=results_df,
    x="fingerprint_size",
    y="Test R2",
    hue="model",    # All entries are XGB, but this can help if you add more models.
    kind="line",
    col="Radius",   # One facet per radius.
    col_wrap=4,     # Adjust number of columns per row.
    marker="o",
    height=4,
    aspect=1.2
)

g.fig.suptitle("Test R2 vs Fingerprint Size by Radius (XGBoost)", y=1.03)
g.savefig('test_r2_by_model_and_radius.png', dpi=300, bbox_inches='tight')
plt.show()



Evaluating: Radius = 1, Fingerprint Size = 10


C:\Users\Fahad\AppData\Local\Temp\ipykernel_45332\1176038021.py:56: DeprecationWarning: The binary mode of fromstring is deprecated, as it behaves surprisingly on unicode inputs. Use frombuffer instead
  X = np.vstack([np.fromstring(fp, 'u1') - ord('0') for fp in fps]).astype(float)


Shape (X, y): ((2220, 10), (2220,))
Shape (X_train, y_train): ((1665, 10), (1665,))
Shape (X_test, y_test): ((555, 10), (555,))
Fitting 5 folds for each of 16 candidates, totalling 80 fits


BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.